In [1]:
import openai, json, requests

client = openai.OpenAI()

messages = []

MOVIE_API_BASE_URL = "https://nomad-movies.nomadcoders.workers.dev/movies"

In [2]:
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "get_popular_movies",
            "description": "A function to get popular movie titles and ID",
            "parameters": {
                "type": "object",
                "properties": {},
                "required": [],
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_movie_details",
            "description": "A function to get movie details by ID",
            "parameters": {
                "type": "object",
                "properties": {
                    "id": {
                        "type": "string",
                        "description": "The movie ID to get details",
                    }
                },
                "requried": ["id"],
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_similar_movies",
            "description": "A function to get similar movies by ID",
            "parameters": {
                "type": "object",
                "properties": {
                    "id": {
                        "type": "string",
                        "description": "The movie ID to get similar movies",
                    }
                },
                "requried": ["id"],
            }
        }
    }
]

In [3]:
def get_popular_movies():
    url = MOVIE_API_BASE_URL
    
    try:
        res = requests.get(url)
        data = res.json()
        
        movies = [
            {
                "title": obj["title"],
                "id": obj["id"]
            } for obj in data
        ]

        return f"현재 인기 목록입니다. {movies}"
    except Exception as err:
        return f"영화 정보를 가져올 수 없습니다. {str(err)}"


def get_movie_details(id):
    url = f"{MOVIE_API_BASE_URL}/{id}"

    try:
        res = requests.get(url)
        data = res.json()

        return json.dumps(data)
    except Exception as err:
        return f"영화 정보를 가져올 수 없습니다. {str(err)}"


def get_similar_movies(id):
    url = f"{MOVIE_API_BASE_URL}/{id}/similar"

    try:
        res = requests.get(url)
        data = res.json()

        movies = [
            {
                "title": obj["title"],
                "id": obj["id"]
            } for obj in data
        ]

        return f"해당 영화와 유사한 영화 목록입니다. {movies}"
    except Exception as err:
        return f"영화 정보를 가져올 수 없습니다. {str(err)}"

FUNCTION_MAP = {
    'get_popular_movies': get_popular_movies,
    'get_movie_details': get_movie_details,
    'get_similar_movies': get_similar_movies,
}

In [4]:
from openai.types.chat import ChatCompletionMessage

def process_ai_response(message: ChatCompletionMessage):
    if message.tool_calls:
        messages.append(
            {
                "role": "assistant",
                "content": message.content or "",
                "tool_calls": [
                    {
                        "id": tool_call.id,
                        "type": "function",
                        "function": {
                            "name": tool_call.function.name,
                            "arguments": tool_call.function.arguments,
                        },
                    }
                    for tool_call in message.tool_calls
                ],
            }
        )
    
        for tool_call in message.tool_calls:
            function_name = tool_call.function.name
            arguments = tool_call.function.arguments

        print(f"AI: Calling function {function_name} with {arguments}")

        try:
            arguments = json.loads(arguments)
        except json.JSONDecodeError:
            arguments = {} 
        
        function_to_run = FUNCTION_MAP.get(function_name)
        result = function_to_run(**arguments)

        messages.append(
            {
            "role": "tool",
            "tool_call_id": tool_call.id,
            "name": function_name,
            "content": result
            }
        )

        call_ai()
    else:
        messages.append({"role": "assistant", "content": message.content})
        print(f"AI: {message.content}")


def call_ai():
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages,
        tools=TOOLS,
    )
    process_ai_response(response.choices[0].message)

In [5]:
while True:
    message = input("Send a message to the LLM...")
    if message == "quit" or message == "q":
        break
    else:
        messages.append(
        {
            "role": "user",
            "content": message,
        }
    )
    print(f"User: {message}")
    call_ai()

User: 인기있는 영화 추천해줘
AI: Calling function get_popular_movies with {}
AI: 다음은 현재 인기 있는 영화 목록입니다:

1. **Your Heart Will Be Broken** (ID: 1523145)
2. **Avatar: Fire and Ash** (ID: 83533)
3. **Hoppers** (ID: 1327819)
4. **The Super Mario Bros. Movie** (ID: 502356)
5. **Shelter** (ID: 1290821)
6. **Crime 101** (ID: 1171145)
7. **Greenland 2: Migration** (ID: 840464)
8. **Project Hail Mary** (ID: 687163)
9. **GOAT** (ID: 1297842)
10. **The Super Mario Galaxy Movie** (ID: 1226863)
11. **Mike & Nick & Nick & Alice** (ID: 1115544)
12. **Sniper: No Nation** (ID: 1641319)
13. **The Bride!** (ID: 1159831)
14. **War Machine** (ID: 1265609)
15. **The Mortuary Assistant** (ID: 1470130)
16. **Pretty Lethal** (ID: 1084187)
17. **Humint** (ID: 1268127)
18. **Scream 7** (ID: 1159559)
19. **The Passion of the Christ** (ID: 615)
20. **Starbright** (ID: 662707)

관심 있는 영화가 있으면 더 알아볼 수 있습니다!
User: 영화 프로젝트 헤일메리에 대해서 더 알려줄래?
AI: Calling function get_movie_details with {"id":"687163"}
AI: **영화: 프로젝트 헤일메리 (Project 